# Kiro에서 Registry를 MCP 도구로 호출하기

#### 코드를 한 줄도 작성하지 않고 Kiro IDE에서 조직 전체의 AWS Agent Registry에 등록된 에이전트 기능을 검색합니다.

## 개요
Kiro와 같은 IDE에서 조직 전체에 구축된 에이전트 기능의 Registry를 직접 검색할 수 있으면 매우 유용합니다. 이 예제에서는 안전하고 별도 설정이 필요 없는 에이전트 및 도구 검색을 위해 Auth0 Dynamic Client Registration(DCR)으로 AWS Agent Registry를 구성하는 방법을 보여 줍니다.

**Dynamic Client Registration이란 무엇인가요?** 

Dynamic Client Registration은 클라이언트 애플리케이션을 미리 수동 등록하지 않고 authorization server에 자동으로 등록할 수 있게 하는 OAuth 및 OpenID Connect 프로토콜입니다. DCR 프로토콜은 RFC 7591에 정식으로 정의되어 있으며, 선택적 등록 관리 확장은 RFC 7592에 정의되어 있습니다. Open Authorization(OAuth) 및 OpenID Connect(OIDC) 생태계에서 작동하도록 설계되었으며, client ID, secret, metadata(redirect URI, scope)를 자동으로 생성할 수 있어 자동화, AI 에이전트 및 동적 확장에 자주 사용됩니다.

IDE에서 AWS Registry MCP를 사용할 때 Dynamic Client Registration을 적용하면 수동 설정 없이 IDE가 AWS Registry 액세스를 위한 고유 자격 증명을 자동으로 받을 수 있습니다. 따라서 IDE가 자체 access token을 프로그래밍 방식으로 요청하고 갱신하므로 사용자가 직접 복사하여 붙여 넣을 필요가 없습니다.

![Kiro MCP JSON](images/0_authflow_dcr.png)

### 주요 기능
- CUSTOM_JWT authorizer로 보호되는 Auth0 기반 AWS Agent Registry 생성
- 샘플 에이전트 레코드로 Registry 초기화
- OAuth bearer token을 사용하여 레코드 검색
- IDE에서 바로 검색할 수 있도록 Kiro에 Registry를 MCP 서버로 연결

## 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:---|:---|
| 튜토리얼 유형 | 대화형 |
| AgentCore 구성 요소 | AWS Agent Registry |
| 인증 방식 | Auth0 DCR (Dynamic Client Registration) |
| 인바운드 인증 IdP | Auth0 (CUSTOM_JWT) |
| 아웃바운드 인증 | OAuth Bearer Token |
| 튜토리얼 구성 | Auth0 DCR Server 생성, OAuth 검색, Kiro의 MCP 설정 |
| 튜토리얼 분야 | 분야 공통 |
| 예제 난이도 | 쉬움 |
| 사용 SDK | boto3 |



## 사전 요구 사항

- tenant가 구성된 **Auth0 계정**
- `boto3`, `python-dotenv`, `requests`가 설치된 **Python 3.10 이상**
- MCP 통합 단계에 사용할 **Kiro/Claude IDE**
- `.env.example`을 기반으로 구성한 `.env` 파일



_______________________________________

# 1단계: Auth0 설정




### 1. Auth0 계정 및 Tenant 생성

1. 계정이 없다면 [auth0.com](https://auth0.com)에서 가입합니다.
2. authorization server 역할을 할 새 tenant를 생성하거나 기존 tenant를 사용합니다.
3. **Auth0 Domain**(예: `your-tenant.auth0.com`)을 기록합니다.
4. Dashboard > Settings > Advanced로 이동하여 Dynamic Client Registration(DCR)을 활성화합니다.

### 2. API(Resource Server) 등록

1. Auth0 Dashboard에서 **Applications → APIs**로 이동합니다.
2. **Create API**를 선택합니다.
3. **Name**을 입력합니다(예: `AWS Agent Registry API`).
4. **Identifier (Audience)**를 다음 URL로 설정합니다: `https://bedrock-agentcore.us-west-2.amazonaws.com` 
5. **Allow Skipping User Consent**를 `true`로 설정합니다(선택 사항이지만 테스트할 때 편리함).
6. **Signing Algorithm**을 선택합니다. `RS256`을 권장합니다.
7. 보안 모범 사례를 적용합니다.
        - **Token Lifetime**을 3600초(1시간) 이하로 설정합니다.
        - **Settings → Advanced Settings → Grant Types**로 이동하여 **Authorization Code, Refresh Token, Client Credentials**를 활성화합니다.
8. API에 할당된 사용자 목록에 사용자를 추가합니다. 이 사용자는 Registry 검색용 access token을 생성하는 데 사용됩니다. **User Management** > **Users** > **Add a user**로 이동합니다.
  

> **⚠️ 주의:** DCR은 클라이언트의 동적 등록을 지원하므로, 연결되는 각 MCP 클라이언트(예: Kiro)가 자동 생성한 서드 파티 클라이언트가 Auth0 tenant에 표시됩니다. Dynamic Client Registration을 효과적으로 사용하려면 보안 영향과 대규모 배포 시의 운영 측면을 모두 고려해야 합니다. DCR이 등록을 자동화하므로 신뢰할 수 있는 클라이언트만 등록할 수 있고 등록 워크플로가 악용되지 않도록 해야 합니다.


### 3. Domain 수준 인증 활성화

DCR이 Auth0에 새 client app을 생성하면 기본적으로 활성화된 connection이 없습니다. Auth0에는 클라이언트가 사용할 수 있는 connection(로그인 방식)을 지정해야 합니다.
1. Auth0 Dashboard에서 **Authentication → Database → Username-Password-Authentication → Settings**로 이동하여 **Promote Connection to Domain Level**을 활성화합니다.


### 4. `.env` 구성

`.env.example`을 `.env`로 복사하고 값을 입력합니다.

### 5. Registry 생성 및 샘플 레코드 등록
이 Notebook의 2단계와 3단계에 따라 Registry를 생성하고 샘플 레코드를 등록합니다.


### 6. Registry의 OAuth Allowed Audience에 AWS Agent Registry API 추가
Kiro는 Auth0 authorization 요청에서 MCP 서버 URL을 `audience`로 전송합니다. 이 값이 없으면 서비스를 찾을 수 없습니다. 다음 두 단계로 진행합니다.
1. MCP URL을 identifier로 사용하는 API를 Auth0에 생성합니다.
2. 해당 API를 allowed audience로 허용하도록 Registry의 OAuth 구성을 업데이트합니다.

이 실습에서는 다음과 같이 진행합니다. 
1. 생성된 Registry ID를 기록합니다. Auth0 Dashboard에서 **Applications → APIs**로 이동한 후 Registry의 MCP endpoint URL을 **Identifier**로 사용하여 API를 생성합니다.
MCP URL: `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp`

2. 이 실습에서는 Registry의 인증 구성을 수동으로 업데이트할 필요가 없습니다. 이 Notebook의 `create_registry` helper는 Registry 생성 후 `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp`를 Registry의 `allowedAudience`에 자동으로 추가합니다. 



# 2단계: Auth0 CUSTOM_JWT Authorizer를 사용하여 Registry 생성

Auth0를 OAuth identity provider로 사용하도록 구성된 새 AWS Agent Registry를 생성합니다. Helper는 다음 작업을 수행합니다.
1. Auth0 tenant의 OIDC discovery URL을 가리키는 `CUSTOM_JWT` authorizer로 Registry 생성
2. Registry가 `READY` 상태가 될 때까지 폴링
3. MCP endpoint URL을 Registry의 `allowedAudience`에 자동으로 추가


In [ ]:
!pip install python-dotenv strands-agents bedrock-agentcore bedrock-agentcore-starter-toolkit

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))
from seed_records import create_registry, seed

registry = create_registry(
    name="auth0-demo-registry-dcr",
    description="Demo registry with Auth0 OAuth for notebook walkthrough",
)
registry_id = registry["registryId"]
print(f"Registry ID: {registry_id}")
print(f"Status: {registry['status']}")

# 3단계: 샘플 기능 레코드를 Registry에 등록


Registry에 4개의 샘플 에이전트 레코드(weather, order-status, customer-support, inventory-lookup)를 등록합니다. 각 레코드는 `DRAFT`로 생성되고 승인 요청을 거쳐 자동 승인된 후 검색 결과에 표시됩니다.



In [ ]:
records = seed(registry_id=registry_id)
print(f"\nSeeded {len(records)} records:")
for r in records:
    print(f"  • {r['name']} ({r['recordId']})")

In [ ]:
from seed_records import _cp_client

cp = _cp_client()
for r in cp.list_registries()["registries"]:
    if r["name"] == "auth0-demo-registry-dcr":
        print(f"Registry: {r['registryId']}")
        recs = cp.list_registry_records(registryId=r["registryId"]).get("registryRecords", [])
        for rec in recs:
            print(f"{rec['recordId']:15s} {rec['status']:10s} {rec['name']}")

# 4단계: MCP URL을 Identifier로 사용하는 API를 Auth0에 추가 생성




Kiro는 Auth0 authorization 요청에서 MCP 서버 URL을 `audience`로 전송합니다. 이 값이 없으면 서비스를 찾을 수 없습니다. 다음 두 단계로 진행합니다.
1. MCP URL을 identifier로 사용하는 API를 Auth0에 생성합니다.
2. 해당 API를 allowed audience로 허용하도록 Registry의 OAuth 구성을 업데이트합니다.

위에서 생성한 `registry_id`를 사용하여 MCP URL `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp`을 만듭니다.

- [수행할 작업] Auth0 Dashboard에서 **Applications → APIs**로 이동한 후 Registry의 MCP endpoint URL을 **Identifier**로 사용하여 API를 생성합니다.
- 이 실습에서는 Notebook의 `create_registry` helper가 Registry 생성 후 `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp`를 Registry의 `allowedAudience`에 자동으로 추가합니다. 여기서는 별도 작업이 필요하지 않습니다.



# 5단계: Kiro에 AWS Agent Registry MCP Server 추가




이제 Registry에서 OAuth 검색이 작동하므로 IDE에서 바로 검색할 수 있도록 Kiro에 MCP 서버로 추가합니다.

### 4.1 `mcp.json`에 추가

Kiro MCP 구성 파일(`.kiro/settings/mcp.json`)에 다음 내용을 추가합니다.

```json
{ "mcpServers" : {
"dcr-registry-server_xx": {
      "type": "http",
      "url": "https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<registry_id>/mcp/",
      "disabled": false
    }
}
}

```
![Kiro MCP JSON](images/1_kiro_mcp_json.png)

### 4.2 인증

Kiro/Claude에서 MCP를 활성화하고 인증합니다.
Kiro가 MCP 서버에 연결되면 다음 작업을 수행합니다.
- Registry의 well-known endpoint를 통해 Auth0 authorization server 검색
- DCR을 사용하여 OAuth client로 자동 등록(POST /oidc/register)
- PKCE authorization code flow를 통해 access token 획득
- Token을 사용하여 Registry 검색 MCP 호출

| Authorization PKCE | 인증 성공 |
|:---:|:---:|
| ![Authorization PKCE](images/2_authorization_pkce.png) | ![Successful Auth](images/3_successful_auth.png) |


### 4.3 Kiro에서 검색
Kiro 채팅을 열고 Registry 검색을 요청합니다.

"Use the AWS registry to search for weather records"

![Kiro MCP JSON](images/4_kiro_search.png) 

### 유용한 참고 명령

Authorization server metadata를 확인합니다.

```
curl -X POST https://<domain url>/oidc/register\
    -H "Content-Type: application/json" \
    -d '{
      "client_name": "test-mcp-client",
      "redirect_uris": ["http://localhost:65358/callback"],
      "grant_types": ["authorization_code"],
      "response_types": ["code"],
      "token_endpoint_auth_method": "none"
    }'

```

DCR을 통해 클라이언트를 수동으로 등록합니다.

```
https://<domainurl>/.well-known/oauth-authorization-server
```





## 정리

In [ ]:
# Registry 및 레코드 정리

from seed_records import delete_registry

delete_registry(registry_id)